# Methods for improving linear regression

In this exercise we will compare various techniques that allow us to achieve a better result for linear regression on the Boston Housing dataset.

Recall that the model from chapter 3, trained only on the variables RM and LSTAT, achieved R² ≈ 0.65.

We will gradually try these approaches:
* **Baseline** – the model from chapter 3 (RM + LSTAT)
* **All variables** – a linear model without any adjustments
* **Correlation + VIF** – multicollinearity diagnostics, a model without collinear variables
* **RFE** – automatic variable selection (Recursive Feature Elimination)
* **EFA** – dimensionality reduction via latent factors
* **PCA** – transformation into uncorrelated components
* **Regularization** – Ridge, Lasso, Elastic Net with cross-validation for selecting λ

At the end we will compare all the models in a clear table.

## Loading and analyzing the data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

In [ ]:
data = pd.read_csv ("..\\dataset\\HousingData.csv")

In [ ]:
data=data.dropna()

In [ ]:
data.head()

In [ ]:
data.describe()

## Baseline model: RM and LSTAT

As our starting point, we will use the model from chapter 3, where based on correlation analysis we selected the variables **RM** (average number of rooms) and **LSTAT** (percentage of the population with lower socioeconomic status).

We also define a helper function `print_model_score` and a list `results`, into which we will continuously store the results of all the models for the final comparison.

We will split the data into a training (80%) and test (20%) part. We set `random_state=42` for reproducibility — the same split will be used across all models.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

results = []

def print_model_score(y_true, y_pred, label=""):
    """Print R² and RMSE; return (r2, rmse)."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    prefix = f"{label} " if label else ""
    print(f"{prefix}R²:   {r2:.4f}")
    print(f"{prefix}RMSE: {rmse:.4f}")
    return r2, rmse

X_base = data[['RM', 'LSTAT']].values
y_base = data['MEDV'].values
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

lr_base = LinearRegression()
lr_base.fit(X_train_base, y_train_base)

r2_tr, _ = print_model_score(y_train_base, lr_base.predict(X_train_base), "Train")
r2_te, rmse_te = print_model_score(y_test_base, lr_base.predict(X_test_base), "Test")
results.append({"Model": "Linear regression (RM+LSTAT)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Linear model using all variables

For comparison, we will build a model that uses all 13 available variables without any adjustments.

In [ ]:
X = np.array(data.drop('MEDV', axis=1))
y = np.array(data['MEDV'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Creating the linear model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

Evaluating the model on the training data

In [ ]:
y_pred = lr.predict(X_train)
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Evaluating the model on the test data

In [ ]:
y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_all_tr = r2_score(y_train, lr.predict(X_train))
r2_all_te = r2_score(y_test, lr.predict(X_test))
rmse_all_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
results.append({
    "Model": "Linear regression (all variables)",
    "Train R²": round(r2_all_tr, 4),
    "Test R²": round(r2_all_te, 4),
    "Test RMSE": round(rmse_all_te, 4),
})

## Correlation
We will again perform correlation analysis and look for linear dependencies between the variables.

In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111)
sns.heatmap(data.corr(),annot=True)

The **correlation matrix** contains the Pearson correlation coefficients between all pairs of variables. The values range from -1 to 1.
* 1 → strong positive linear correlation
* -1 → strong negative linear correlation
* 0 → no linear correlation

**Multicollinearity** occurs when the predictor variables are strongly correlated with each other (e.g. DIS with INDUS, NOX, AGE).

The problem:
* The estimates of the regression coefficients are unstable and sensitive to small changes in the data.
* The interpretation of individual coefficients loses meaning, because the effect of one variable cannot be isolated while holding the others fixed, if they are correlated with each other.
* It increases the standard error of the coefficients, which reduces statistical significance.


The important row is the last one, which shows us the linear correlation of the explanatory variables with the explained variable MEDV. It seems that our target variable is highly correlated with LSTAT and RM, which makes sense, since these two factors are very important for determining house prices. There is also a lot of multicollinearity here.

The usual interpretation of a regression coefficient is that it provides an estimate of the effect of a unit change in the independent variable on the dependent variable, while holding the other variables constant. In the case of multicollinearity, however, we cannot say this. If X1 in a given dataset is strongly correlated with another independent variable, X2, then we have a set of observations for which X1 and X2 have a certain linear stochastic relationship. So we cannot guarantee that X2 will remain constant when X1 changes.

## Variance Inflation Factor (VIF) 

The **Variance Inflation Factor (VIF)** detects multicollinearity in a regression analysis.

Its presence can negatively affect the regression results. VIF estimates how much the variance of a regression coefficient is inflated due to multicollinearity in the model.

VIF=1/(1−R^2)

Where R^2 is the coefficient of determination.

Put simply, it is the proportion of the variance of the independent variable that is explained by the dependent variable.

So we perform linear regression using each variable as the target and the others as predictors, and calculate R^2, and then calculate VIF from it.

If VIF < 4, the variable can be used; otherwise we must find a way to remove the collinearity of these variables.

In [ ]:
vifdf = []
for i in data.columns:
    X = np.array(data.drop(i,axis=1))
    y = np.array(data[i])
    lr = LinearRegression()
    lr.fit(X,y)
    y_pred = lr.predict(X)
    r2 = r2_score(y,y_pred)
    vif = 1/(1-r2)
    vifdf.append((i,vif))

vifdf = pd.DataFrame(vifdf,columns=['Features','Variance Inflation Factor'])
vifdf.sort_values(by='Variance Inflation Factor')

We can see that almost half of the variables have a VIF value higher than, or close to, 4. TAX and RAD have a VIF almost twice as high as our threshold value.

So it will be a good idea to address the multicollinearity. This can be done in several ways:
* Removing correlated variables → keep only one from a pair of strongly correlated variables.
* Principal Component Analysis (PCA) → transform the predictors into uncorrelated components.
* Regularization (Ridge, Lasso) → suppresses the influence of collinearity and stabilizes the model.

### Model without collinear variables

We will remove the variables with too high a VIF (TAX, RAD) and train a new linear model. This will let us verify whether reducing multicollinearity alone improves the prediction.

In [ ]:
features_novif = [c for c in data.columns if c not in ['MEDV', 'TAX', 'RAD']]
X_novif = data[features_novif].values
X_train_novif, X_test_novif, y_train_novif, y_test_novif = train_test_split(
    X_novif, data['MEDV'].values, test_size=0.2, random_state=42
)

lr_novif = LinearRegression()
lr_novif.fit(X_train_novif, y_train_novif)

r2_tr, _ = print_model_score(y_train_novif, lr_novif.predict(X_train_novif), "Train")
r2_te, rmse_te = print_model_score(y_test_novif, lr_novif.predict(X_test_novif), "Test")
results.append({"Model": "Linear regression (wihout TAX+RAD)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Recursive Feature Elimination (RFE)

RFE is a method for automatic variable selection. It works iteratively:
1. Trains a model on all the variables.
2. Removes the variable with the smallest absolute coefficient.
3. Repeats until the desired number of variables remains.

Unlike VIF (which addresses collinearity), RFE selects variables based on their **contribution to the prediction**.

In [ ]:
from sklearn.feature_selection import RFE

feature_names = [c for c in data.columns if c != 'MEDV']
X_rfe_raw = data[feature_names].values
X_train_rfe, X_test_rfe, y_train_rfe, y_test_rfe = train_test_split(
    X_rfe_raw, data['MEDV'].values, test_size=0.2, random_state=42
)

rfe = RFE(LinearRegression(), n_features_to_select=6)
rfe.fit(X_train_rfe, y_train_rfe)

selected_features = [feature_names[i] for i, s in enumerate(rfe.support_) if s]
print("Selected variables:", selected_features)

lr_rfe = LinearRegression()
lr_rfe.fit(X_train_rfe[:, rfe.support_], y_train_rfe)

r2_tr, _ = print_model_score(y_train_rfe, lr_rfe.predict(X_train_rfe[:, rfe.support_]), "Train")
r2_te, rmse_te = print_model_score(y_test_rfe, lr_rfe.predict(X_test_rfe[:, rfe.support_]), "Test")
results.append({"Model": "Linear regression + RFE (6 variables)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## Exploratory Factor Analysis (EFA)

EFA is a dimensionality reduction method that looks for hidden (latent) factors explaining the correlation structure among the variables.

Comparison with PCA:
* **PCA** maximizes the explained variance of the data — the components are linear combinations of the variables.
* **EFA** models the correlation structure and assumes that the observed variables are caused by a smaller number of hidden factors.

We will use 3 factors with **varimax** rotation (the factors remain uncorrelated), just like in chapter 2.

We standardize using only the training data (**fit on train, transform on test**), so that no information leaks from the test set (data leakage).

In [ ]:
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

feature_names_efa = [c for c in data.columns if c != 'MEDV']
X_efa_raw = data[feature_names_efa].values
X_train_efa_raw, X_test_efa_raw, y_train_efa, y_test_efa = train_test_split(
    X_efa_raw, data['MEDV'].values, test_size=0.2, random_state=42
)

scaler_efa = StandardScaler()
X_train_efa_sc = scaler_efa.fit_transform(X_train_efa_raw)
X_test_efa_sc = scaler_efa.transform(X_test_efa_raw)

fa = FactorAnalysis(n_components=3, rotation='varimax', random_state=42)
X_train_fa = fa.fit_transform(X_train_efa_sc)
X_test_fa = fa.transform(X_test_efa_sc)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feature_names_efa,
    columns=['Factor 1', 'Factor 2', 'Factor 3']
).round(3)
loadings

### Linear model based on EFA factors

We will use the factor scores (3 values instead of 13 variables) as the input for linear regression.

In [ ]:
lr_efa = LinearRegression()
lr_efa.fit(X_train_fa, y_train_efa)

r2_tr, _ = print_model_score(y_train_efa, lr_efa.predict(X_train_fa), "Train")
r2_te, rmse_te = print_model_score(y_test_efa, lr_efa.predict(X_test_fa), "Test")
results.append({"Model": "Linear regression + EFA (3 factors)", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

## PCA

### The idea
* If we have many mutually correlated variables, there is hidden redundancy in the data.
* PCA removes this redundancy by converting the original variables into new, uncorrelated variables = principal components.
* These components are a linear combination of the original variables.


### How PCA works (intuition)
* It finds the direction with the greatest variance in the data (1st principal component).
* It finds a second direction with the greatest possible variance, but orthogonal to the first (2nd principal component).
* It continues until it exhausts all dimensions.
* Result:
    * The principal components are uncorrelated.
    * The first components explain most of the variability in the data.


### What PCA is used for
* Removing multicollinearity → the components are orthogonal → no collinearity.
* Dimensionality reduction → we keep only the first few components, which explain e.g. 90–95% of the variability.
* Visualization → complex data with many variables can be plotted in 2D/3D space.


PCA is sensitive to the scale of the variables. That's why standardization is usually done before applying PCA.

We will not write PCA by hand, but will use its implementation from the library.

In [ ]:
from sklearn.decomposition import PCA

The number of PCA components will be 13, the same as the number of input parameters.

We must remove the output MEDV from the input to PCA.

The data must be standardized.

In [ ]:
feature_names_pca = [c for c in data.columns if c != 'MEDV']
X_pca_raw = data[feature_names_pca].values
scaler_pca = StandardScaler()
X_pca_sc = scaler_pca.fit_transform(X_pca_raw)

In [ ]:
pca = PCA(n_components=13)
X_pca = pca.fit_transform(X_pca_sc)

In [ ]:
X_pca_all = pd.DataFrame(X_pca,columns=['PCA1','PCA2','PCA3','PCA4','PCA5','PCA6','PCA7','PCA8','PCA9','PCA10','PCA11','PCA12','PCA13'])

The distribution functions of the PCA variables are different from the original ones.

In [ ]:
pos = 1
fig = plt.figure(figsize=(12,16))
for i in X_pca_all.columns:
    ax = fig.add_subplot(7,2, pos)
    pos = pos + 1
    sns.histplot(X_pca_all[i],ax=ax, kde=True)

PCA was supposed to reduce multicollinearity. So let's check it.

The correlation matrix shows that the PCA components are not dependent on each other — this is the key property of PCA.

**Why doesn't MEDV correlate the most with PCA1?**
PCA orders the components by how much variance of the **input variables X** they explain — not by correlation with the target variable MEDV. That's why the first component explains the most variability in the data, but it may not best predict MEDV.

A model trained on the first 6 components (in PCA order, i.e. highest explained variance) may therefore not be optimal. If we wanted to optimize the selection for MEDV, we would use a different method (e.g. PLS — Partial Least Squares).

In [ ]:
data_pca_all = pd.DataFrame(
    X_pca,
    columns=[f'PCA{i}' for i in range(1, 14)]
)
data_pca_all['MEDV'] = y

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111)
sns.heatmap(data_pca_all.corr(), annot=True)

## Linear model from all PCA variables 
Now we will create a new dataset with the principal components as input variables and MEDV as the output variable.

We will again split the data into a training and test part.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_all.values, y, test_size=0.2, random_state=42
)

We will create a linear model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

Validating the model on the training data

In [ ]:
y_pred = lr.predict(X_train)
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Validating the model on the test data

In [ ]:
y_pred = lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_pca_tr = r2_score(y_train, lr.predict(X_train))
r2_pca_te = r2_score(y_test, lr.predict(X_test))
rmse_pca_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))
results.append({
    "Model": "Linear regression + PCA (13 components)",
    "Train R²": round(r2_pca_tr, 4),
    "Test R²": round(r2_pca_te, 4),
    "Test RMSE": round(rmse_pca_te, 4),
})

The resulting model from the PCA variables is somewhat better than the original linear model built from the original variables.

## Linear model from 6 PCA variables
PCA can also be used for dimensionality reduction.

So we will build a model that has only 6 input variables instead of 13.

In [ ]:
lr = LinearRegression()
lr.fit(X_train[:, :6], y_train)

Validating the model on the training data

In [ ]:
y_pred = lr.predict(X_train[:,0:6])
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

Validating the model on the test data

In [ ]:
y_pred = lr.predict(X_test[:,0:6])
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
r2_pca6_tr = r2_score(y_train, lr.predict(X_train[:, :6]))
r2_pca6_te = r2_score(y_test, lr.predict(X_test[:, :6]))
rmse_pca6_te = np.sqrt(mean_squared_error(y_test, lr.predict(X_test[:, :6])))
results.append({
    "Model": "Linear regression + PCA (6 components)",
    "Train R²": round(r2_pca6_tr, 4),
    "Test R²": round(r2_pca6_te, 4),
    "Test RMSE": round(rmse_pca6_te, 4),
})

As expected, the accuracy of the reduced model is somewhat lower. On the other hand, the model is smaller.

## Regularization

Regularization adds a penalty term to the linear regression loss function. This suppresses excessively large coefficients and reduces the risk of overfitting.

| Method | Penalty | Effect |
|--------|-----------|-------|
| **Ridge (L2)** | sum of squared coefficients | Keeps all variables, shrinks their influence |
| **Lasso (L1)** | sum of absolute values of coefficients | Zeroes out insignificant coefficients → automatic variable selection |
| **Elastic Net** | combination of L1 + L2 | `l1_ratio` determines the ratio; suitable for groups of correlated variables |

The amount of penalization is determined by the parameter **λ** (in scikit-learn, `alpha`). We find the optimal value using cross-validation.

**Important:** Regularization is sensitive to scale — we must standardize the data first. We fit the scaler only on the training set.

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_names_reg = [c for c in data.columns if c != 'MEDV']
X_reg = data[feature_names_reg].values
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, data['MEDV'].values, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_train_sc = scaler_reg.fit_transform(X_train_reg)
X_test_sc = scaler_reg.transform(X_test_reg)

### Ridge (L2 regularization)

`RidgeCV` searches over the given `alpha` values and selects the best one using cross-validation.

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha: {ridge.alpha_:.4f}")
r2_tr, _ = print_model_score(y_train_reg, ridge.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, ridge.predict(X_test_sc), "Test")
results.append({"Model": "Ridge CV", "Train R²": r2_tr, "Test R²": r2_te, "Test RMSE": rmse_te})

### Lasso (L1 regularization)

Lasso can zero out the coefficients of insignificant variables — it implicitly performs variable selection. `LassoCV` searches for the optimal `alpha` using cross-validation.

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(alphas=np.logspace(-3, 1, 100), cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha: {lasso.alpha_:.4f}")
print(f"Non-zero coeficients: {np.sum(lasso.coef_ != 0)} / {X_train_sc.shape[1]}")
r2_tr, _ = print_model_score(y_train_reg, lasso.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, lasso.predict(X_test_sc), "Test")
results.append({"Model": "Lasso CV", 
                "Train R²": r2_tr, 
                "Test R²": r2_te, 
                "Test RMSE": rmse_te})

### Elastic Net

Elastic Net combines L1 and L2 penalization. `ElasticNetCV` searches over combinations of `alpha` and `l1_ratio` (the ratio of L1 to L2 — 0 = Ridge, 1 = Lasso) using cross-validation.

In [ ]:
from sklearn.linear_model import ElasticNetCV

elastic = ElasticNetCV(
    alphas=np.logspace(-3, 1, 50),
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    cv=5,
    random_state=42,
    max_iter=10000,
)
elastic.fit(X_train_sc, y_train_reg)

print(f"Optimal alpha:    {elastic.alpha_:.4f}")
print(f"Optimal l1_ratio: {elastic.l1_ratio_:.2f}")
r2_tr, _ = print_model_score(y_train_reg, elastic.predict(X_train_sc), "Train")
r2_te, rmse_te = print_model_score(y_test_reg, elastic.predict(X_test_sc), "Test")
results.append({"Model": "Elastic Net CV", "Train R²": r2_tr, "Test R²": r2_te, "Test RMSE": rmse_te})

## Comparison of the models

The results of all the models, sorted by test R². A higher R² and lower RMSE mean a better prediction on new data.

Models that use more variables are more accurate on the test data than simplified models using only some of the data.

In [ ]:
pd.DataFrame(results).round(4).sort_values("Test R²", ascending=False).reset_index(drop=True)